## Act 2: The Raw City

Objective:
To evaluate the integrity of raw NYC taxi trip data by identifying inconsistencies,
defining what constitutes a "real trip", and analyzing how different definitions
impact analytical conclusions.

Key Question:
What is a valid trip, and how does redefining validity affect system behavior insights?

In [1]:
import pandas as pd

files = [
    "yellow_tripdata_2023-02.parquet",
    "yellow_tripdata_2023-03.parquet",
    "yellow_tripdata_2023-04.parquet",
    "yellow_tripdata_2023-05.parquet",
    "yellow_tripdata_2023-06.parquet"
]

cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "tip_amount"
]

df = pd.concat([
    pd.read_parquet(f, columns=cols).sample(frac=0.2)
    for f in files
], ignore_index=True)

df.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,tip_amount
0,2023-02-01 20:14:36,2023-02-01 20:26:38,1.86,12.8,3.56
1,2023-02-14 20:03:15,2023-02-14 20:11:28,1.02,9.3,2.14
2,2023-02-22 20:47:21,2023-02-22 21:02:57,3.44,18.4,4.68
3,2023-02-02 11:04:56,2023-02-02 11:07:40,0.61,5.8,1.20
4,2023-02-04 14:02:19,2023-02-04 14:13:20,1.58,12.1,1.61


In [2]:
#creating base features
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df['trip_duration'] = (
    (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime'])
    .dt.total_seconds() / 60
)

df['month'] = df['tpep_pickup_datetime'].dt.month

In [3]:
#detecting data problems 
# Invalid conditions
zero_distance = (df['trip_distance'] == 0).sum()
negative_fare = (df['fare_amount'] <= 0).sum()
invalid_duration = (df['trip_duration'] <= 0).sum()
high_duration = (df['trip_duration'] > 120).sum()
tip_issue = (df['tip_amount'] > df['fare_amount']).sum()

print("Zero distance:", zero_distance)
print("Negative/zero fare:", negative_fare)
print("Invalid duration:", invalid_duration)
print("Too long trips:", high_duration)
print("Tip > Fare:", tip_issue)

Zero distance: 44782
Negative/zero fare: 30620
Invalid duration: 1300
Too long trips: 3860
Tip > Fare: 33419


## Defining a Real Trip

A trip is considered valid if:

1. trip_distance > 0
2. fare_amount > 0
3. trip_duration between 1 and 120 minutes
4. pickup time < dropoff time
5. tip_amount <= fare_amount

This definition ensures logical and operational consistency.

In [4]:
#applying filtering 
df_clean = df[
    (df['trip_distance'] > 0) &
    (df['fare_amount'] > 0) &
    (df['trip_duration'] > 1) &
    (df['trip_duration'] < 120) &
    (df['tip_amount'] <= df['fare_amount'])
]

print("Original:", df.shape)
print("Cleaned:", df_clean.shape)

Original: (3285371, 7)
Cleaned: (3197091, 7)


In [5]:
#comparing before vs after cleaning
def summarize(data, name):
    print(f"\n--- {name} ---")
    print("Trips:", len(data))
    print("Avg Distance:", data['trip_distance'].mean())
    print("Avg Fare:", data['fare_amount'].mean())
    print("Avg Duration:", data['trip_duration'].mean())
    print("Tip %:", (data['tip_amount']/data['fare_amount']).mean()*100)

summarize(df, "RAW DATA")
summarize(df_clean, "CLEANED DATA")


--- RAW DATA ---
Trips: 3285371
Avg Distance: 4.159701400541978
Avg Fare: 19.274484884659906
Avg Duration: 17.231262562634985
Tip %: inf

--- CLEANED DATA ---
Trips: 3197091
Avg Distance: 4.224111437553701
Avg Fare: 19.483600207188346
Avg Duration: 16.160073886959534
Tip %: 20.265595136631365


## Final Conclusion

The definition of a "real trip" significantly influences analytical outcomes.

Strict validation removes anomalous records, leading to more reliable insights,
while relaxed definitions may exaggerate or distort system behavior.

Thus, data cleaning is not merely a preprocessing step but a critical decision-making process
that shapes the interpretation of system stability.